In [2]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from deep_translator import GoogleTranslator
import re
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/Deep Learning/Recipe Generation/cuisines.csv")

In [5]:
df.head(2)

,name,image_url,description,cuisine,course,diet,prep_time,ingredients,instructions
0,Thayir Semiya Recipe (Curd Semiya),https://www.archanaskitchen.com/images/archana...,Thayir Semiya or Curd Vermicelli is a quick di...,Indian,Lunch,Vegetarian,Total in 35 M,\n\n\t\t\t\t\t\t\t \t\t1/2 cup Semiya (Vermice...,"To begin making the Thayir Semiya recipe, firs..."
1,Chettinad Style Kara Kuzhambu Recipe with Pota...,https://www.archanaskitchen.com/images/archana...,Chettinad Style Kara Kuzhambu Recipe with Pot...,South Indian Recipes,Lunch,Vegetarian,Total in 75 M,\nFor ground masala\n\n\t\t\t\t\t\t\t \t\t1/4 ...,To begin making the Chettinad Style Kara Kuzha...


In [6]:
df['instructions'][0]

'To begin making the Thayir Semiya recipe, firstly boil the vermicelli in 2 cups water. Drain it and spread on a colander.Pass cold water through it. This ensures that the vermicelli does not become sticky. Keep it aside and let it cool down.Heat a kadai/wok and add oil. Add the peanuts and roast till it turns brownish. Remove and keep it aside.In the same kadai, add more oil if needed and add the mustard seeds and after it splutters add the other ingredients meant for tempering.Add the boiled vermicelli and mix everything. Add salt to taste and mix.Finally just before serving add curd and mix well.\xa0Serve Thayir Semiya on its own for a light and healthy lunch or dinner'

In [8]:

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

def translate_to_english(text):

    if pd.isna(text):
        return "Not Available"

    text = str(text)

    try:
        translated = GoogleTranslator(
            source='auto',
            target='en'
        ).translate(text)

        return translated

    except:
        return text

df['text'] = (
    df['name'].fillna('') + " " +
    df['description'].fillna('') + " " +
    df['ingredients'].fillna('')
)

df['text'] = df['text'].apply(clean_text)



tfidf = TfidfVectorizer(max_features=500)

X = tfidf.fit_transform(df['text']).toarray()



target_column = "cuisine"   # change if needed

encoder = LabelEncoder()
y = encoder.fit_transform(df[target_column])



X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)



model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(np.unique(y)), activation='softmax')
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32
)


loss, accuracy = model.evaluate(X_test, y_test)

print("\nAccuracy:", accuracy)

y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



model.save("/content/drive/MyDrive/Deep Learning/Recipe Generation/recipe_model.keras")
model.save_weights(
    "/content/drive/MyDrive/Deep Learning/Recipe Generation/recipe_model.weights.h5"
)

with open("/content/drive/MyDrive/Deep Learning/Recipe Generation/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("/content/drive/MyDrive/Deep Learning/Recipe Generation/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("\n✅ Model Saved Successfully")

Epoch 1/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.1708 - loss: 3.3916 - val_accuracy: 0.1799 - val_loss: 2.7574
Epoch 2/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2557 - loss: 2.7263 - val_accuracy: 0.3732 - val_loss: 2.4643
Epoch 3/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3483 - loss: 2.4364 - val_accuracy: 0.3791 - val_loss: 2.2203
Epoch 4/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3945 - loss: 2.1923 - val_accuracy: 0.4336 - val_loss: 2.0586
Epoch 5/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4642 - loss: 1.9801 - val_accuracy: 0.4912 - val_loss: 1.9123
Epoch 6/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5203 - loss: 1.8254 - val_accuracy: 0.5177 - val_loss: 1.7979
Epoch 7/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5542 - loss: 1.6785 - val_accuracy: 0.5428 - val_loss: 1.7290
Epoch 8/100
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5845 - loss: 1.5772 - val_accuracy: 0.5472 - 

In [9]:
user_input = input("\nEnter Recipe Description (or type exit): ")

cleaned_text = clean_text(user_input)

    # Convert to TF-IDF
X_input = tfidf.transform([cleaned_text]).toarray()

    # Predict
prediction = model.predict(X_input)

predicted_index = np.argmax(prediction)

predicted_cuisine = encoder.inverse_transform([predicted_index])

confidence = np.max(prediction) * 100

print("\n🍽 Predicted Cuisine:", predicted_cuisine[0])
print("📊 Confidence:", round(confidence, 2), "%")



from sklearn.metrics.pairwise import cosine_similarity


recipes = df[df['cuisine'] == predicted_cuisine[0]]


recipe_vectors = tfidf.transform(recipes['text'])


similarities = cosine_similarity(X_input, recipe_vectors)


best_index = similarities.argmax()

recipe = recipes.iloc[best_index]



print("\n==============================")
print("🍴 BEST MATCHING RECIPE")
print("==============================")

print("\n🍽 Name:")
print(translate_to_english(recipe['name']))

print("\n📝 Description:")
print(translate_to_english(recipe['description']))

print("\n🥘 Ingredients:")
print(translate_to_english(recipe['ingredients']))

if 'instructions' in df.columns:
    print("\n👨‍🍳 Instructions:")
    print(translate_to_english(recipe['instructions']))

if 'prep_time' in df.columns:
    print("\n⏰ Prep Time:")
    print(translate_to_english(recipe['prep_time']))

if 'cook_time' in df.columns:
    print("\n⏲ Cook Time:")
    print(translate_to_english(recipe['cook_time']))


Enter Recipe Description (or type exit): how to make paneer masala
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step

🍽 Predicted Cuisine: North Indian Recipes
📊 Confidence: 97.01 %

🍴 BEST MATCHING RECIPE

🍽 Name:
Achari Paneer Masala Recipe

📝 Description:
Achari Paneer Masala Recipe is a flavourful gravy that you can make for your house parties. Pickled spices are used in it which makes it even more delicious. It is easy to make and because of the cheese in it, the amount of protein is also high.

🥘 Ingredients:
250 grams cheese 

							 		2 onions, finely chopped							 	

							 		4 tomatoes 

							 		1 capsicum (green) 

							 		4 cashews 

							 		1 tbsp mustard oil 

							 		1 green chilli 

Asafoetida, a pinch							 	

salt, as per taste							 	
for spices

							 		1 tablespoon coriander seeds 

							 		1 tablespoon fennel 

							 		1 tsp mustard 

							 		1/2 tsp fenugreek seeds 

							 		2 dried Kashmiri chillies  

							 		1/4 tsp nigella seeds  

							 		1 